# Entrenamiento de modelos (Penguins)

Notebook para entrenar modelos y guardarlos versionados en `models/`.

- `ModelType.GMM` -> guarda en `models/gmm/`
- `ModelType.RF`  -> guarda en `models/rf/`

El versionado detecta el ultimo `vN.pkl` existente y crea `v(N+1).pkl`.


## Imports y configuracion

In [1]:
import json
import os
import re
from enum import Enum
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    silhouette_score,
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
CSV_PATH = "penguins.csv"
MODELS_DIR = Path("models")

# Features supervisado (RF)
NUMERIC_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL_FEATURES = ["island", "sex"]

# Features no supervisado (GMM)
GMM_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
N_CLUSTERS = 3  # Adelie, Chinstrap, Gentoo

In [3]:
class ModelType(Enum):
    GMM = 1
    RF = 2

## Versionado

Busca la ultima version `vN.pkl` en `models/<tipo>/` y devuelve la ruta de la siguiente version.

In [4]:
def get_next_version_path(model_type: ModelType) -> Path:
    """Devuelve la ruta del siguiente vN.pkl para el tipo de modelo dado.

    Escanea models/<tipo>/ buscando archivos con patron v<numero>.pkl,
    toma el numero mas alto y devuelve v(max+1).pkl. Si no hay ninguno,
    empieza en v1.pkl.
    """
    subdir = MODELS_DIR / model_type.name.lower()
    subdir.mkdir(parents=True, exist_ok=True)

    pattern = re.compile(r"^v(\d+)\.pkl$")
    versions = [
        int(m.group(1))
        for f in subdir.iterdir()
        if (m := pattern.match(f.name))
    ]
    next_version = (max(versions) + 1) if versions else 1
    return subdir / f"v{next_version}.pkl"


def save_model(payload, model_type: ModelType) -> Path:
    """Guarda el payload como el siguiente vN.pkl y devuelve la ruta usada."""
    path = get_next_version_path(model_type)
    joblib.dump(payload, path)
    print(f"Modelo guardado en: {path}")
    return path

## Carga de datos

In [5]:
def load_data(path: str = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "rowid" in df.columns:
        df = df.drop(columns=["rowid"])
    return df


df_raw = load_data()
print(f"Filas: {len(df_raw)}")
df_raw.head()

Filas: 344


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## Entrenamiento RF (supervisado)

Limpia NA, codifica categoricas y target, entrena varios candidatos y se queda con el mejor por accuracy. Guarda modelos, encoders y metadata dentro de un unico `vN.pkl`.

In [6]:
def train_rf(df: pd.DataFrame):
    # Limpieza: elimino filas con NA (son pocas)
    df = df.dropna().reset_index(drop=True).copy()

    # Codificacion de categoricas de entrada + target
    encoders = {}
    for col in CATEGORICAL_FEATURES:
        enc = LabelEncoder()
        df[col] = enc.fit_transform(df[col])
        encoders[col] = enc

    species_encoder = LabelEncoder()
    df["species"] = species_encoder.fit_transform(df["species"])
    encoders["species"] = species_encoder

    feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES
    X = df[feature_columns]
    y = df["species"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    models = {
        "logistic_regression": Pipeline(
            [("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))]
        ),
        "decision_tree": DecisionTreeClassifier(random_state=42),
        "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    }

    accuracies = {}
    species_names = encoders["species"].classes_

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        accuracies[name] = acc
        print(f"Modelo '{name}' -> accuracy: {acc:.4f}")
        print(classification_report(
            y_test, y_pred,
            labels=range(len(species_names)),
            target_names=species_names,
        ))

    best_model = max(accuracies, key=accuracies.get)
    print(f"Mejor modelo: '{best_model}' ({accuracies[best_model]:.4f})")

    metadata = {
        "feature_columns": feature_columns,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "available_models": list(models.keys()),
        "default_model": best_model,
        "accuracies": accuracies,
    }

    payload = {
        "models": models,
        "encoders": encoders,
        "metadata": metadata,
    }
    return payload


rf_payload = train_rf(df_raw)
save_model(rf_payload, ModelType.RF)

Modelo 'logistic_regression' -> accuracy: 1.0000
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        29
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        24

    accuracy                           1.00        67
   macro avg       1.00      1.00      1.00        67
weighted avg       1.00      1.00      1.00        67

Modelo 'decision_tree' -> accuracy: 0.9403
              precision    recall  f1-score   support

      Adelie       0.96      0.93      0.95        29
   Chinstrap       0.82      1.00      0.90        14
      Gentoo       1.00      0.92      0.96        24

    accuracy                           0.94        67
   macro avg       0.93      0.95      0.94        67
weighted avg       0.95      0.94      0.94        67

Modelo 'random_forest' -> accuracy: 1.0000
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00     

PosixPath('models/rf/v3.pkl')

## Entrenamiento GMM (no supervisado)

Imputa, escala y ajusta un GaussianMixture de 3 componentes. Mapea cada cluster a la especie mayoritaria y guarda el pipeline + el mapeo en `vN.pkl`.

In [8]:
def train_gmm(df: pd.DataFrame):
    df = df.dropna(subset=GMM_FEATURES).copy()
    X = df[GMM_FEATURES]

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("gmm", GaussianMixture(n_components=N_CLUSTERS, random_state=42, n_init=10)),
    ])

    labels = pipeline.fit_predict(X)
    print(f"Muestras usadas: {len(X)}")
    print(f"Silhouette score: {silhouette_score(X, labels):.3f}")

    cluster_to_species = None
    if "species" in df.columns:
        ari = adjusted_rand_score(df["species"], labels)
        print(f"Adjusted Rand Index vs especie real: {ari:.3f}")

        cluster_to_species = (
            df.assign(cluster=labels)
            .groupby("cluster")["species"]
            .agg(lambda s: s.mode()[0])
            .to_dict()
        )
        print(f"Mapeo cluster -> especie: {cluster_to_species}")

        predicted_species = [cluster_to_species[c] for c in labels]
        accuracy = accuracy_score(df["species"], predicted_species)
        print(f"Accuracy (especie mayoritaria por cluster): {accuracy:.3f}")

    payload = {
        "pipeline": pipeline,
        "cluster_to_species": cluster_to_species,
        "features": GMM_FEATURES,
    }
    return payload


gmm_payload = train_gmm(df_raw)
save_model(gmm_payload, ModelType.GMM)

Muestras usadas: 342
Silhouette score: 0.145
Adjusted Rand Index vs especie real: 0.960
Mapeo cluster -> especie: {0: 'Gentoo', 1: 'Adelie', 2: 'Chinstrap'}
Accuracy (especie mayoritaria por cluster): 0.985
Modelo guardado en: models/gmm/v4.pkl


PosixPath('models/gmm/v4.pkl')

## Uso rapido

Para entrenar y versionar segun tipo:

```python
save_model(train_rf(df_raw), ModelType.RF)    # -> models/rf/vN.pkl
save_model(train_gmm(df_raw), ModelType.GMM)  # -> models/gmm/vN.pkl
```

Cada llamada a `save_model` crea automaticamente la siguiente version.